#  Telco Customer Churn — Full EDA & Feature Engineering Notebook
**Dataset:** WA_Fn-UseC_-Telco-Customer-Churn.csv  
**Goal:** Load, clean, explore, engineer features, encode, and split data for modeling.

## 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

# ── Plot style ──────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

DATA_PATH = '../data/WA_Fn-UseC_-Telco-Customer-Churn.csv'
print('Libraries loaded ')

Libraries loaded 


## 2.  Load Data & Basic Info

In [2]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
print('=== .info() ===')
df.info()

=== .info() ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non

In [4]:
print('=== .describe() — numeric ===')
df.describe()

=== .describe() — numeric ===


,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [5]:
print('=== .describe() — categorical ===')
df.describe(include='object')

=== .describe() — categorical ===


,customerID,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,TotalCharges,Churn
count,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043
unique,7043,2,2,2,2,3,3,3,3,3,3,3,3,3,2,4,6531,2
top,7590-VHVEG,Male,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,,No
freq,1,3555,3641,4933,6361,3390,3096,3498,3088,3095,3473,2810,2785,3875,4171,2365,11,5174


## 3.  Missing Values & Data Types

In [6]:
# ── Check raw dtypes and missing values ─────────────────────────────────────
missing = df.isnull().sum()
pct     = (missing / len(df) * 100).round(2)
dtype   = df.dtypes
summary = pd.DataFrame({'dtype': dtype, 'missing': missing, 'missing_%': pct})
print(summary[summary['missing'] > 0])
print('\nAll columns:')
summary

Empty DataFrame
Columns: [dtype, missing, missing_%]
Index: []

All columns:


,dtype,missing,missing_%
customerID,object,0,0.0
gender,object,0,0.0
SeniorCitizen,int64,0,0.0
Partner,object,0,0.0
Dependents,object,0,0.0
tenure,int64,0,0.0
PhoneService,object,0,0.0
MultipleLines,object,0,0.0
InternetService,object,0,0.0
OnlineSecurity,object,0,0.0


## 4.  Data Cleaning

In [7]:
# ── 4a. TotalCharges: blank spaces → NaN → numeric ──────────────────────────
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

n_miss = df['TotalCharges'].isnull().sum()
print(f'TotalCharges — NaN after coerce: {n_miss}')

# Inspect the rows (all are new customers with tenure == 0)
df[df['TotalCharges'].isnull()][['customerID','tenure','MonthlyCharges','TotalCharges']]

TotalCharges — NaN after coerce: 11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,NaN
753,3115-CZMZD,0,20.25,NaN
936,5709-LVOEQ,0,80.85,NaN
1082,4367-NUYAO,0,25.75,NaN
1340,1371-DWPAZ,0,56.05,NaN
3331,7644-OMVMY,0,19.85,NaN
3826,3213-VVOLG,0,25.35,NaN
4380,2520-SGTTA,0,20.00,NaN
5218,2923-ARZLG,0,19.70,NaN
6670,4075-WKNIU,0,73.35,NaN


In [8]:
# ── 4b. Fill missing TotalCharges with 0 (tenure==0 → no charges yet) ───────
# Alternatively use median: df['TotalCharges'].fillna(df['TotalCharges'].median())
df['TotalCharges'] = df['TotalCharges'].fillna(0)
print(f'Remaining NaN in TotalCharges: {df["TotalCharges"].isnull().sum()}')

Remaining NaN in TotalCharges: 0


In [9]:
# ── 4c. Encode target: Churn → binary ───────────────────────────────────────
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print('Churn value counts:')
print(df['Churn'].value_counts())

Churn value counts:
Churn
0    5174
1    1869
Name: count, dtype: int64


In [10]:
# ── 4d. Drop useless column ──────────────────────────────────────────────────
df.drop(columns=['customerID'], inplace=True)
print(f'Shape after dropping customerID: {df.shape}')

Shape after dropping customerID: (7043, 20)


In [11]:
# ── Final check ──────────────────────────────────────────────────────────────
print('Remaining NaN per column:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\nAll good ✅' if df.isnull().sum().sum() == 0 else '⚠️ Still has NaN')

Remaining NaN per column:
Series([], dtype: int64)

All good ✅


## 5. 🔎 Exploratory Data Analysis (EDA)

### 5.1 Overall Churn Rate

In [ ]:
churn_rate = df['Churn'].mean() * 100
print(f'Overall churn rate: {churn_rate:.2f}%')

fig, ax = plt.subplots(figsize=(5, 4))
counts = df['Churn'].value_counts()
wedge_props = dict(width=0.5, edgecolor='white', linewidth=2)
ax.pie(
    counts,
    labels=['No Churn', 'Churn'],
    autopct='%1.1f%%',
    colors=['#4C72B0', '#DD8452'],
    wedgeprops=wedge_props,
    startangle=90,
    textprops={'fontsize': 12}
)
ax.set_title('Customer Churn Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.2 Univariate Analysis — Numerical Features

In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()
num_cols.remove('Churn')   # target handled separately
print('Numerical columns:', num_cols)

fig, axes = plt.subplots(1, len(num_cols), figsize=(5 * len(num_cols), 4))
if len(num_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, num_cols):
    df[col].hist(bins=40, ax=ax, color='#4C72B0', edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
plt.suptitle('Numerical Feature Distributions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 5.3 Univariate Analysis — Categorical Features

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'{len(cat_cols)} categorical columns:', cat_cols)

n_cols = 3
n_rows = (len(cat_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, ax=axes[i], order=order, palette='muted')
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30)
    for p in axes[i].patches:
        axes[i].annotate(f'{int(p.get_height())}',
                         (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='bottom', fontsize=9)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Categorical Feature Distributions', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 5.4 Bivariate Analysis — Churn vs Key Features

In [ ]:
# ── Churn vs Contract ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

churn_by_contract = df.groupby('Contract')['Churn'].mean().sort_values(ascending=False) * 100
churn_by_contract.plot(kind='bar', ax=axes[0], color=['#DD8452','#4C72B0','#55A868'], edgecolor='white')
axes[0].set_title('Churn Rate by Contract Type', fontweight='bold')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_xlabel('')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].tick_params(axis='x', rotation=15)

sns.countplot(data=df, x='Contract', hue='Churn', ax=axes[1],
              palette={0: '#4C72B0', 1: '#DD8452'})
axes[1].set_title('Count by Contract & Churn', fontweight='bold')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(title='Churn', labels=['No', 'Yes'])

plt.tight_layout()
plt.show()

In [ ]:
# ── Churn vs Tenure ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(data=df, x='tenure', hue='Churn', ax=axes[0],
             bins=30, palette={0: '#4C72B0', 1: '#DD8452'}, alpha=0.7)
axes[0].set_title('Tenure Distribution by Churn', fontweight='bold')
axes[0].legend(title='Churn', labels=['No', 'Yes'])

sns.boxplot(data=df, x='Churn', y='tenure', ax=axes[1],
            palette={0: '#4C72B0', 1: '#DD8452'})
axes[1].set_title('Tenure Boxplot by Churn', fontweight='bold')
axes[1].set_xticklabels(['No Churn', 'Churn'])

plt.tight_layout()
plt.show()

In [ ]:
# ── Churn vs MonthlyCharges ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(data=df, x='MonthlyCharges', hue='Churn', ax=axes[0],
             bins=30, palette={0: '#4C72B0', 1: '#DD8452'}, alpha=0.7)
axes[0].set_title('MonthlyCharges Distribution by Churn', fontweight='bold')
axes[0].legend(title='Churn', labels=['No', 'Yes'])

sns.boxplot(data=df, x='Churn', y='MonthlyCharges', ax=axes[1],
            palette={0: '#4C72B0', 1: '#DD8452'})
axes[1].set_title('MonthlyCharges Boxplot by Churn', fontweight='bold')
axes[1].set_xticklabels(['No Churn', 'Churn'])

plt.tight_layout()
plt.show()

In [ ]:
# ── Churn rate across all categorical features ────────────────────────────────
cat_features = [c for c in cat_cols if c != 'Churn']
n_cols = 3
n_rows = (len(cat_features) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    churn_rate_col = df.groupby(col)['Churn'].mean().sort_values(ascending=False) * 100
    churn_rate_col.plot(kind='bar', ax=axes[i], color='#DD8452', edgecolor='white')
    axes[i].set_title(f'Churn Rate by {col}', fontweight='bold')
    axes[i].set_ylabel('Churn Rate (%)')
    axes[i].set_xlabel('')
    axes[i].yaxis.set_major_formatter(mtick.PercentFormatter())
    axes[i].tick_params(axis='x', rotation=30)
    # Annotate bars
    for p in axes[i].patches:
        axes[i].annotate(f'{p.get_height():.1f}%',
                         (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='bottom', fontsize=8)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Churn Rate by Categorical Feature', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Churn vs InternetService ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.countplot(data=df, x='InternetService', hue='Churn', ax=axes[0],
              palette={0: '#4C72B0', 1: '#DD8452'})
axes[0].set_title('Count by Internet Service & Churn', fontweight='bold')
axes[0].legend(title='Churn', labels=['No', 'Yes'])

churn_by_is = df.groupby('InternetService')['Churn'].mean().sort_values(ascending=False) * 100
churn_by_is.plot(kind='bar', ax=axes[1], color='#4C72B0', edgecolor='white')
axes[1].set_title('Churn Rate by Internet Service', fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# ── Churn vs PaymentMethod ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.countplot(data=df, x='PaymentMethod', hue='Churn', ax=axes[0],
              palette={0: '#4C72B0', 1: '#DD8452'})
axes[0].set_title('Count by Payment Method & Churn', fontweight='bold')
axes[0].legend(title='Churn', labels=['No', 'Yes'])
axes[0].tick_params(axis='x', rotation=30)

churn_by_pm = df.groupby('PaymentMethod')['Churn'].mean().sort_values(ascending=False) * 100
churn_by_pm.plot(kind='bar', ax=axes[1], color='#55A868', edgecolor='white')
axes[1].set_title('Churn Rate by Payment Method', fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

### 5.5 Correlation Heatmap — Numerical Features

In [ ]:
num_df = df.select_dtypes(include='number')

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(num_df.corr(), dtype=bool))
sns.heatmap(
    num_df.corr(),
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    mask=mask,
    ax=ax,
    linewidths=0.5,
    vmin=-1, vmax=1,
    square=True
)
ax.set_title('Correlation Heatmap — Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. 🛠️ Feature Engineering

In [ ]:
df_fe = df.copy()

# ── TenureGroup: bin tenure into customer lifecycle stages ───────────────────
bins   = [0, 12, 24, 48, 60, 72]
labels = ['0-1yr', '1-2yr', '2-4yr', '4-5yr', '5-6yr']
df_fe['TenureGroup'] = pd.cut(df_fe['tenure'], bins=bins, labels=labels, include_lowest=True)

# ── TotalServices: count number of add-on services subscribed to ─────────────
service_cols = ['PhoneService', 'MultipleLines', 'InternetService',
                'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
# Map Yes=1, No=0, No internet/phone service=0
df_fe['TotalServices'] = df_fe[service_cols].apply(
    lambda col: col.map(lambda x: 1 if x == 'Yes' else 0)
).sum(axis=1)

# ── AvgMonthlyCharge: TotalCharges / (tenure+1) — avoid division by zero ─────
df_fe['AvgMonthlyCharge'] = df_fe['TotalCharges'] / (df_fe['tenure'] + 1)

# ── ChargeRatio: MonthlyCharges / (TotalCharges+1) — price trend indicator ───
df_fe['ChargeRatio'] = df_fe['MonthlyCharges'] / (df_fe['TotalCharges'] + 1)

# ── IsSeniorWithPartner: interaction — senior citizen with a partner ──────────
df_fe['IsSeniorWithPartner'] = (
    (df_fe['SeniorCitizen'] == 1) & (df_fe['Partner'] == 'Yes')
).astype(int)

# ── HasPaperlessBilling (already boolean-like) ───────────────────────────────
# PaperlessBilling is already Yes/No — will be encoded later

print('New features created:')
print(df_fe[['TenureGroup','TotalServices','AvgMonthlyCharge',
             'ChargeRatio','IsSeniorWithPartner']].describe())

In [ ]:
# ── Visualise new features vs Churn ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# TenureGroup
tg_churn = df_fe.groupby('TenureGroup')['Churn'].mean().sort_index() * 100
tg_churn.plot(kind='bar', ax=axes[0], color='#4C72B0', edgecolor='white')
axes[0].set_title('Churn Rate by Tenure Group', fontweight='bold')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].tick_params(axis='x', rotation=30)

# TotalServices
ts_churn = df_fe.groupby('TotalServices')['Churn'].mean() * 100
ts_churn.plot(kind='bar', ax=axes[1], color='#DD8452', edgecolor='white')
axes[1].set_title('Churn Rate by # of Services', fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].tick_params(axis='x', rotation=0)

# AvgMonthlyCharge
sns.boxplot(data=df_fe, x='Churn', y='AvgMonthlyCharge', ax=axes[2],
            palette={0: '#4C72B0', 1: '#DD8452'})
axes[2].set_title('Avg Monthly Charge by Churn', fontweight='bold')
axes[2].set_xticklabels(['No Churn', 'Churn'])

plt.suptitle('Engineered Features vs Churn', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. 🔠 Encode Categorical Variables

In [ ]:
df_enc = df_fe.copy()

# ── Binary columns: Yes/No → 1/0 ─────────────────────────────────────────────
binary_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService',
    'PaperlessBilling'
]
for col in binary_cols:
    df_enc[col] = df_enc[col].map({'Yes': 1, 'No': 0,
                                   'Male': 1, 'Female': 0})

# ── TenureGroup: ordinal encode ───────────────────────────────────────────────
tg_order = {'0-1yr': 0, '1-2yr': 1, '2-4yr': 2, '4-5yr': 3, '5-6yr': 4}
df_enc['TenureGroup'] = df_enc['TenureGroup'].map(tg_order)

# ── Multi-class categoricals: One-Hot Encode ──────────────────────────────────
ohe_cols = [
    'MultipleLines', 'InternetService', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod'
]
df_enc = pd.get_dummies(df_enc, columns=ohe_cols, drop_first=False)

# ── Convert bool columns produced by get_dummies → int ───────────────────────
bool_cols = df_enc.select_dtypes(include='bool').columns
df_enc[bool_cols] = df_enc[bool_cols].astype(int)

print(f'Shape after encoding: {df_enc.shape}')
df_enc.head(3)

In [ ]:
# ── Verify no object columns remain (except TenureGroup already mapped) ───────
remaining_obj = df_enc.select_dtypes(include='object').columns.tolist()
print('Remaining object columns:', remaining_obj if remaining_obj else 'None ✅')

## 8. ✂️ Train / Test Split (80 / 20, Stratified)

In [ ]:
X = df_enc.drop(columns=['Churn'])
y = df_enc['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f'Train size : {X_train.shape[0]:,} rows  ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test size  : {X_test.shape[0]:,} rows  ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'\nChurn rate — train : {y_train.mean()*100:.2f}%')
print(f'Churn rate — test  : {y_test.mean()*100:.2f}%')
print('\nStratification preserved ✅')

## 9. 💾 Save Outputs

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

# ── Cleaned dataset (post cleaning, pre encoding) ─────────────────────────────
df_fe.to_csv('../data/processed/telco_churn_cleaned.csv', index=False)
print('Saved: telco_churn_cleaned.csv')

# ── Fully processed / encoded dataset ─────────────────────────────────────────
df_enc.to_csv('../data/processed/telco_churn_encoded.csv', index=False)
print('Saved: telco_churn_encoded.csv')

# ── Train / Test splits ───────────────────────────────────────────────────────
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv',  index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv',  index=False)

print('Saved: X_train.csv, X_test.csv, y_train.csv, y_test.csv')
print('\n✅ All files saved to ../data/processed/')

## 10. 📋 Summary

| Step | Detail |
|------|--------|
| Raw shape | 7043 rows × 21 columns |
| Cleaned | Dropped `customerID`, fixed `TotalCharges`, encoded `Churn` |
| Missing values | 11 rows with blank `TotalCharges` → filled with 0 |
| New features | `TenureGroup`, `TotalServices`, `AvgMonthlyCharge`, `ChargeRatio`, `IsSeniorWithPartner` |
| Encoding | Binary map + OHE for multi-class categoricals |
| Split | 80/20 stratified, churn rate preserved |
| Saved files | `telco_churn_cleaned.csv`, `telco_churn_encoded.csv`, `X_train/test.csv`, `y_train/test.csv` |

**Key EDA findings:**
- Overall churn rate ≈ **26.5%** — moderately imbalanced.
- **Month-to-month** contracts churn far more than 1- or 2-year contracts.
- Customers with **Fiber Optic** internet churn the most.
- **Electronic check** payers have the highest churn rate of all payment methods.
- Churn drops significantly with **tenure** — the first 12 months are the highest-risk window.
- Higher **MonthlyCharges** correlate with churn; low-charge customers are more loyal.
- Customers using **more services** actually churn less (bundle stickiness effect).